# AML fraud/laundering detection: full GPU training

Runs the pipeline from https://github.com/Sumanasn/aml-tgn-fraud-detection on Kaggle's free GPU.

**Before running:** Settings (right panel) -> Accelerator: **GPU T4 x2** -> Internet: On.

Avoid the plain P100 accelerator here -- it's older Pascal-generation hardware, and newer
PyPI `torch` wheels have been dropping compiled kernels for it, which surfaces as
`CUDA error: no kernel image is available for execution on the device` the moment you run
anything on GPU. T4 (Turing, compute capability 7.5) doesn't have this problem.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
print('torch:', torch.__version__, '| built for CUDA:', torch.version.cuda)
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print(f'GPU compute capability: {major}.{minor}')
    if (major, minor) < (7, 0):
        print('WARNING: compute capability < 7.0 (e.g. P100) -- switch Settings -> Accelerator to GPU T4 x2 to avoid "no kernel image available" errors after installing torch-geometric.')

In [ ]:
!git clone https://github.com/Sumanasn/aml-tgn-fraud-detection.git
%cd aml-tgn-fraud-detection

In [ ]:
# Kaggle ships torch pre-built and matched to its GPU already. A plain
# `pip install torch-geometric` can pull in a newer torch as a transitive
# dependency, silently replacing that working build with a wheel that may
# not ship CUDA kernels for this GPU (this is what causes "no kernel image
# is available for execution on the device" on P100 in particular).
# --no-deps keeps the existing torch untouched; torch_geometric core
# (SAGEConv/TransformerConv/JumpingKnowledge/TGNMemory, everything this
# project uses) doesn't require torch-scatter/torch-sparse to run.
!pip install -q --no-deps torch-geometric
!pip install -q xgboost

import torch, torch_geometric
print('torch (after install):', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('pyg:', torch_geometric.__version__)
assert torch.cuda.is_available(), "torch lost CUDA after the pip install -- see the markdown cell above for the fix."

# Sanity check: actually run something on the GPU now, so a kernel-image
# mismatch fails here with a clear message instead of 20 minutes into training.
torch.randn(4, 4, device='cuda').sum()
print('GPU kernel smoke test passed.')

In [ ]:
!python -m src.data_pipeline.download_amlsim --dataset 20K_cycle200

## 1. XGBoost baseline (tabular + hand-engineered graph features)

In [ ]:
!python -m src.training.train_baseline

## 2. GraphSAGE GNN (10-layer, residual + JumpingKnowledge)

Depth is matched to the ~10-12 hop cycle length found in the fraud subgraph (see `src/models/gnn.py` docstring). GPU makes this fast enough to bump epochs/hidden size if you want to push accuracy further -- edit the call below.

In [ ]:
from src.training.train_gnn import main as train_gnn
train_gnn(epochs=300, patience=30)

## 3. TGN (memory + temporal attention embedding)

This is the one that's genuinely slow on CPU (~1-2 min/epoch locally at batch_size=200,
hundreds of tiny sequential batches). On a T4 this should run substantially faster --
bump `epochs` here now that GPU is available; 3 epochs was only a local correctness smoke test.

In [ ]:
from src.training.train_tgn import main as train_tgn
train_tgn(epochs=10, batch_size=200)

## 4. Time-to-detection: 3-way comparison

XGBoost / GraphSAGE GNN are re-scored on cumulative snapshots; TGN is replayed once, causally.

In [ ]:
from src.training.time_to_detection import main as time_to_detection
time_to_detection()

In [ ]:
from IPython.display import Image
Image(filename='results/time_to_detection.png')

## 5. Copy results + checkpoints to /kaggle/working for download

In [ ]:
import shutil
shutil.copytree('results', '/kaggle/working/results', dirs_exist_ok=True)
shutil.copytree('checkpoints', '/kaggle/working/checkpoints', dirs_exist_ok=True)
print('Copied. Download from the Output tab, or commit the checkpoints back to the repo yourself if you want them versioned.')